In [0]:
from pyspark.sql import functions as F

NAMESPACE = "evhns-agentops-2026"
TOPIC = "agent-traces"
lake = "abfss://bronze@stagentops.dfs.core.windows.net"
conn = dbutils.secrets.get("kv-agentops", "eh-connection-string")

kafka_opts = {
    "kafka.bootstrap.servers": f"{NAMESPACE}.servicebus.windows.net:9093",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="$ConnectionString" password="{conn}";'
    ),
}

schema = """trace_id STRING, event_time STRING, ingest_time STRING, agent_name STRING,
            user_id STRING, model STRING, query_type STRING, prompt_tokens BIGINT,
            completion_tokens BIGINT, tool_calls BIGINT, latency_ms BIGINT,
            status STRING, error_type STRING, region STRING"""

raw = (spark.readStream.format("kafka").options(**kafka_opts)
       .option("subscribe", TOPIC)
       .option("startingOffsets", "earliest")
       .load())

parsed = (raw
    .select(F.from_json(F.col("value").cast("string"), schema).alias("e"),
            F.col("partition").alias("_kafka_partition"),
            F.col("offset").alias("_kafka_offset"),
            F.col("timestamp").alias("_kafka_ts"))
    .select("e.*", "_kafka_partition", "_kafka_offset", "_kafka_ts")
    .withColumn("_ingested_at", F.current_timestamp()))

(parsed.writeStream
    .option("checkpointLocation", f"{lake}/_checkpoints/agent_traces_stream")
    .trigger(availableNow=True)
    .toTable("dbw_agentops.bronze.agent_traces_stream")
    .awaitTermination())

print(spark.table("dbw_agentops.bronze.agent_traces_stream").count(), "rows in bronze stream table")